In [2]:
# =============================================================================
# 1. 文本嵌入（Text Embedding）测试
# =============================================================================
# 作用：将文本转换为高维向量，使计算机能够理解文本的语义
# =============================================================================

import os
import dotenv  # 用于从 .env 文件加载环境变量
from openai import OpenAI

# 待转换的示例文本
input_text = "衣服的质量杠杠的"

# 从 .env 文件加载环境变量（内含 API key）
dotenv.load_dotenv()

# 初始化 OpenAI 客户端（使用阿里云 DashScope 兼容接口）
client = OpenAI(
    api_key=os.getenv("apikey"),  # 从环境变量获取 API key
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"  # 阿里云 API 地址
)

# 调用嵌入模型，将文本转换为向量
completion = client.embeddings.create(
    model="text-embedding-v4",  # 使用阿里云的嵌入模型，默认输出 1024 维向量
    input=input_text
)

print("返回结果 JSON：")

# 将完整结果保存为 JSON 文件（便于调试和查看）
completion_json = completion.model_dump_json(indent=2, ensure_ascii=False)

save_path = "embedding_result.json"
with open(save_path, "w", encoding="utf-8") as f:
    f.write(completion_json)

# 提取核心的嵌入向量（实际开发中常用）
embedding = completion.data[0].embedding
print(f"\n文本 '{input_text}' 的嵌入向量长度：{len(embedding)}")
print(f"嵌入向量前5个值：{embedding[:5]}")

返回结果 JSON：

文本 '衣服的质量杠杠的' 的嵌入向量长度：1024
嵌入向量前5个值：[0.02258586511015892, -0.08700370043516159, -0.013521800749003887, -0.05904024466872215, 0.027100207284092903]


In [38]:
# =============================================================================
# 2. 加载知识库数据
# =============================================================================
# 作用：从本地文件加载猫的知识库，作为后续检索的数据源
# =============================================================================

dataset = []
with open('cat-facts.txt', 'r', encoding='utf-8') as file:
    dataset = file.readlines()  # 按行读取所有内容
    print(f'✅ 加载行数：{len(dataset)}')

✅ 加载行数：150


实现向量数据库:vector_database

In [3]:
# =============================================================================
# 3. 构建向量数据库（Vector Database）
# =============================================================================
# 作用：将知识库的每一句话转换为向量，存储在内存中，用于后续相似度检索
# 核心思想：语义相似的文本，在向量空间中距离更近
# =============================================================================

vector_db = []  # 初始化空列表，存储格式：[(文本, 向量), (文本, 向量), ...]

def add_chunk_to_vector_db(chunk):
    """
    将一段文本转换为向量并添加到数据库
    :param chunk: 待处理的文本片段
    """
    # 调用嵌入模型获取向量
    completion = client.embeddings.create(
        model="text-embedding-v4",
        input=chunk
    )
    embedding = completion.data[0].embedding
    # 同时存储原始文本和对应的向量
    vector_db.append((chunk, embedding))

# 遍历数据集，为每一条知识生成向量
for i, chunk in enumerate(dataset):
    add_chunk_to_vector_db(chunk)

# 将向量数据库保存到本地文件（避免每次重启都要重新生成）
import pickle
with open('vector_db.pkl', 'wb') as f:
    pickle.dump(vector_db, f)

# 打印统计信息
print(f'✅ 向量数据库大小：{len(vector_db)} 条')
print(f'📊 第一个向量前5个值：{vector_db[0][1][:5]}')

NameError: name 'dataset' is not defined

In [4]:
# =============================================================================
# 4. 验证向量数据库
# =============================================================================
# 作用：检查向量数据库的维度是否一致，确保后续计算不会出错
# =============================================================================

import numpy as np
import pickle

# 1. 从本地文件读取向量数据库
with open('vector_db.pkl', 'rb') as f:
    loaded_vector_db = pickle.load(f)

# 2. 提取所有嵌入向量（忽略文本部分）
#    因为所有向量维度一致，可以直接提取元组的第二个元素
embeddings = [item[1] for item in loaded_vector_db]

# 3. 转换为 NumPy 数组（便于后续批量计算）
embedding_array = np.array(embeddings)

# 4. 打印关键信息
print(f"✅ 整个向量库的形状：{embedding_array.shape}")
print(f"📊 向量总数：{embedding_array.shape[0]}，单个向量维度：{embedding_array.shape[1]}")

# 额外验证：确保所有向量维度一致（避免计算时出错）
all_dims = [len(item[1]) for item in loaded_vector_db]
is_all_same = len(set(all_dims)) == 1
print(f"\n✅ 所有 embedding 维度是否一致：{is_all_same}")
if not is_all_same:
    print("⚠️ 发现维度不一致的 embedding，维度分布：", set(all_dims))

✅ 整个向量库的形状：(150, 1024)
📊 向量总数：150，单个向量维度：1024

✅ 所有 embedding 维度是否一致：True


In [5]:
# =============================================================================
# 5. 实现检索功能（RAG 的 "R" 部分 - Retrieval）
# =============================================================================
# 作用：根据用户查询，从向量数据库中找出最相关的知识片段
# 核心方法：余弦相似度（Cosine Similarity）- 衡量两个向量的方向相似度
# =============================================================================

def cos_similarity(vec1, vec2):
    """
    计算两个向量的余弦相似度
    公式：cos(θ) = (A·B) / (||A|| × ||B||)
    返回值范围：[-1, 1]，越接近 1 表示越相似
    """
    # 计算点积（对应位置相乘再相加）
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    # 计算向量范数（长度）
    norm_vec1 = sum(a * a for a in vec1) ** 0.5
    norm_vec2 = sum(b * b for b in vec2) ** 0.5
    # 防止除零错误
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return dot_product / (norm_vec1 * norm_vec2)

def retrieve(query, top_k=3, retriever_top_k=10):
    """
    根据查询文本，检索最相关的 top_k 条知识
    :param query: 用户的问题
    :param top_k: 最终返回的条数
    :param retriever_top_k: 初步检索的条数（用于 reranking）
    :return: 包含 (文本, 相似度) 的列表，按相似度降序排列
    """
    # 1. 将用户问题转换为向量
    query_embedding = client.embeddings.create(
        model="text-embedding-v4",
        input=query
    ).data[0].embedding
    
    # 2. 计算与数据库中每条知识的相似度
    similarities = []
    for chunk, vec in vector_db:
        sim = cos_similarity(query_embedding, vec)
        similarities.append((chunk, sim))
    
    # 3. 按相似度降序排序，最相关的排在前面
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # 4. 返回 top_k 条最相关的知识
    return similarities[:retriever_top_k]

In [9]:
# =============================================================================
# 8. 实现 Cross-Encoder Reranker
# =============================================================================
# 作用：对初步检索的结果进行精确重排序，提升最终结果的相关性
# 
# Cross-Encoder vs Bi-Encoder:
#   - Bi-Encoder (如之前用的): 分别计算 Query 和 Doc 的向量，再用余弦相似度
#     优点: 快速，可预先计算 Doc 向量
#     缺点: 精度较低，无法捕捉 Query 和 Doc 的深层交互
#   
#   - Cross-Encoder: 将 Query 和 Doc 一起输入模型，直接输出相关性分数
#     优点: 精度高，能捕捉深层语义交互
#     缺点: 速度慢，无法预先计算
#
# 因此，常见的做法是：先用 Bi-Encoder 快速检索 top_k (如 100 条)，
# 再用 Cross-Encoder 精确重排序得到最终的 top_k (如 10 条)
# =============================================================================

import pickle

# 加载向量数据库（如果内存中没有）
if 'vector_db' not in globals() or len(vector_db) == 0:
    print("📂 正在从本地加载向量数据库...")
    with open('vector_db.pkl', 'rb') as f:
        vector_db = pickle.load(f)
    print(f"✅ 向量数据库加载完成！共 {len(vector_db)} 条")
else:
    print(f"✅ 向量数据库已存在，共 {len(vector_db)} 条")

from sentence_transformers import CrossEncoder

# 使用本地下载的轻量级模型 (22M 参数，261MB)
model_path = r"D:\sikm\Desktop\PythonProject\AI_Learning\RAG_Learn\RAG_from_Scratch\ms-marco-MiniLM-L6-v2"

print(f"\n⏳ 正在加载 Cross-Encoder 模型...")
print(f"📂 模型路径: {model_path}")

reranker = CrossEncoder(model_path)
print("✅ Cross-Encoder 模型加载完成！")

def rerank(query, candidates, top_k=3):
    """
    使用 Cross-Encoder 对候选结果进行重排序
    :param query: 用户的问题
    :param candidates: 初步检索的候选结果，格式 [(文本, 初始相似度), ...]
    :param top_k: 最终返回的条数
    :return: 重排序后的结果，格式 [(文本, rerank分数), ...]
    """
    # 1. 构建 Cross-Encoder 输入格式：[(query, doc1), (query, doc2), ...]
    pairs = [[query, chunk] for chunk, _ in candidates]
    
    # 2. 调用模型计算每对的相关性分数
    scores = reranker.predict(pairs)
    
    # 3. 组合结果：[(文本, rerank分数), ...]
    reranked_results = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    
    # 4. 按 rerank 分数降序排序
    reranked_results.sort(key=lambda x: x[1], reverse=True)
    
    return reranked_results[:top_k]

# 测试 reranker
test_query = "猫咪一胎能生多少小猫？"
print(f"\n{'='*60}")
print(f"🔍 测试查询: {test_query}")
print(f"{'='*60}\n")

# 先用 Bi-Encoder 初步检索
print("📌 步骤 1: Bi-Encoder 初步检索 (top 10)")
candidates = retrieve(test_query, retriever_top_k=10)
for i, (chunk, score) in enumerate(candidates[:5], 1):
    print(f"  {i}. [{score:.4f}] {chunk[:50]}...")

# 再用 Cross-Encoder 重排序
print(f"\n📌 步骤 2: Cross-Encoder 重排序 (top 3)")
reranked = rerank(test_query, candidates, top_k=3)
for i, (chunk, score) in enumerate(reranked, 1):
    print(f"  {i}. [{score:.4f}] {chunk[:50]}...")

✅ 向量数据库已存在，共 150 条

⏳ 正在加载 Cross-Encoder 模型...
📂 模型路径: D:\sikm\Desktop\PythonProject\AI_Learning\RAG_Learn\RAG_from_Scratch\ms-marco-MiniLM-L6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5621.53it/s]
BertForSequenceClassification LOAD REPORT from: D:\sikm\Desktop\PythonProject\AI_Learning\RAG_Learn\RAG_from_Scratch\ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-Encoder 模型加载完成！

🔍 测试查询: 猫咪一胎能生多少小猫？

📌 步骤 1: Bi-Encoder 初步检索 (top 10)
  1. [0.8420] 大多数猫一胎生1～9只小猫。有记录最多的一胎是**19只**，其中15只存活。
...
  2. [0.7673] 一对猫及其后代，在**7年内理论上能繁殖出42万只小猫**，数量惊人。
...
  3. [0.7598] 如果不干预，母猫**每4个月可生3～7只小猫**。这就是为什么绝育控量非常重要。
...
  4. [0.6961] 有记录生育最多的猫是**达斯蒂（Dusty）**，一生生了超过420只小猫。
...
  5. [0.6478] 猫的孕期约**58～65天**。
...

📌 步骤 2: Cross-Encoder 重排序 (top 3)
  1. [8.0203] 大多数猫一胎生1～9只小猫。有记录最多的一胎是**19只**，其中15只存活。
...
  2. [7.4255] 最高龄生育的猫是**基蒂（Kitty）**，30岁时生下两只小猫，一生共生了218只。
...
  3. [7.3454] 有记录生育最多的猫是**达斯蒂（Dusty）**，一生生了超过420只小猫。
...


In [11]:
# =============================================================================
# 9. 完整 RAG 流程：带 Reranking 的检索 + 生成
# =============================================================================
# 完整流程：用户问题 → Bi-Encoder 检索 → Cross-Encoder 重排序 → LLM 生成回答
# =============================================================================

import pickle

# 确保向量数据库已加载
if 'vector_db' not in globals() or len(vector_db) == 0:
    print("📂 正在从本地加载向量数据库...")
    with open('vector_db.pkl', 'rb') as f:
        vector_db = pickle.load(f)
    print(f"✅ 向量数据库加载完成！共 {len(vector_db)} 条")

# 获取用户输入
input_query = input('请输入你想了解的猫咪知识: ')

# 步骤1：Bi-Encoder 初步检索（获取更多候选）
print(f"\n{'='*60}")
print(f"🔍 步骤 1: Bi-Encoder 初步检索 (top 10)")
print(f"{'='*60}")
candidates = retrieve(input_query, retriever_top_k=10)
for i, (chunk, score) in enumerate(candidates[:5], 1):
    print(f"  {i}. 相似度 [{score:.4f}]: {chunk[:60]}...")

# 步骤2：Cross-Encoder 精确重排序
print(f"\n{'='*60}")
print(f"🎯 步骤 2: Cross-Encoder 精确重排序 (top 3)")
print(f"{'='*60}")
retrieved_knowledge = rerank(input_query, candidates, top_k=3)
for i, (chunk, score) in enumerate(retrieved_knowledge, 1):
    print(f"  {i}. Rerank分数 [{score:.4f}]: {chunk[:60]}...")

# 步骤3：构建 LLM 提示词
context_chunks = '\n'.join([f' - {chunk}' for chunk, score in retrieved_knowledge])

instruction_prompt = f'''你是一个乐于助人的聊天机器人。仅使用以下上下文来回答问题，不要编造任何新信息。:
{context_chunks}'''

print(f'指令提示词：\n{instruction_prompt}')
# 步骤4：调用 LLM 生成回答
print(f"\n{'='*60}")
print(f"🤖 步骤 3: LLM 生成回答")
print(f"{'='*60}\n")

stream = client.chat.completions.create(
    model="qwen-max",
    messages=[
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='', flush=True)

print(f"\n\n{'='*60}")


🔍 步骤 1: Bi-Encoder 初步检索 (top 10)
  1. 相似度 [0.8084]: 照顾得好，猫能活20岁甚至更久，**家猫平均寿命14岁**。
...
  2. 相似度 [0.7435]: 给猫绝育能**延长2～3年寿命**。
...
  3. 相似度 [0.7179]: 有记录最长寿的猫是美国得克萨斯州的**奶油泡芙（Crème Puff）**，活了1967年到2005年8月6日，过完38...
  4. 相似度 [0.6707]: 最高龄生育的猫是**基蒂（Kitty）**，30岁时生下两只小猫，一生共生了218只。
...
  5. 相似度 [0.6643]: 猫咪平均每天会花掉**三分之二**的时间睡觉。这意味着一只9岁的猫，一生中只有3年是清醒的。
...

🎯 步骤 2: Cross-Encoder 精确重排序 (top 3)
  1. Rerank分数 [8.2178]: 有记录最长寿的猫是美国得克萨斯州的**奶油泡芙（Crème Puff）**，活了1967年到2005年8月6日，过完38...
  2. Rerank分数 [7.0294]: 猫咪平均每天会花掉**三分之二**的时间睡觉。这意味着一只9岁的猫，一生中只有3年是清醒的。
...
  3. Rerank分数 [7.0194]: 猫在短距离内的**最高时速约为31英里（约49公里）**。
...
指令提示词：
你是一个乐于助人的聊天机器人。仅使用以下上下文来回答问题，不要编造任何新信息。:
 - 有记录最长寿的猫是美国得克萨斯州的**奶油泡芙（Crème Puff）**，活了1967年到2005年8月6日，过完38岁生日3天后去世。猫通常能活到20岁，相当于人类约96岁。

 - 猫咪平均每天会花掉**三分之二**的时间睡觉。这意味着一只9岁的猫，一生中只有3年是清醒的。

 - 猫在短距离内的**最高时速约为31英里（约49公里）**。


🤖 步骤 3: LLM 生成回答

猫通常能活到20岁，这相当于人类的约96岁。不过，有记录最长寿的猫是美国得克萨斯州的一只名叫奶油泡芙（Crème Puff）的猫，它活了38年。

